# Ejercicio 2.b. Tamaño máximo con coste cúbico

**Sistemas de Big Data · Cuaderno resuelto**

Un AMD Ryzen Threadripper 3990X tiene la tasa facilitada de **2.356.230 MIPS**.
¿Cuál es el mayor tamaño entero de problema que puede abordar durante **un día**
si el algoritmo requiere exactamente $f(n)=n^3$ operaciones?

Ejecuta las celdas en orden. El cuaderno es autónomo, utiliza solo la biblioteca
estándar de Python y no necesita montar Drive ni instalar paquetes.

## 1. Entender las unidades y fijar el modelo

- Interpretamos `2,356,230` como 2.356.230, usando comas como separadores de miles.
- Un MIPS equivale a un millón de instrucciones por segundo.
- El plazo de este ejercicio es un día, aunque la fórmula copiada del ejercicio anterior mencionase un año.
- Para el cálculo idealizado, equiparamos una instrucción de la tasa dada a una operación de $f(n)$.

La frecuencia de 4,35 GHz y el año 2020 identifican los datos del equipo en el enunciado.
No se multiplica la tasa MIPS por la frecuencia ni de nuevo por el número de núcleos.

In [5]:
# Los guiones bajos ayudan a leer el entero sin modificar su valor.
velocidad_mips = 2_356_230

# Conversión de millones de instrucciones/s a operaciones/s según el modelo.
operaciones_por_segundo = velocidad_mips * 1_000_000

# Un día tiene 24 horas, cada hora 60 minutos y cada minuto 60 segundos.
segundos = 24 * 60 * 60
presupuesto = operaciones_por_segundo * segundos

print("Operaciones por segundo:", operaciones_por_segundo)
print("Segundos disponibles:", segundos)
print("Presupuesto de operaciones:", presupuesto)

Operaciones por segundo: 2356230000000
Segundos disponibles: 86400
Presupuesto de operaciones: 203578272000000000


## 2. Plantear la desigualdad

El presupuesto disponible es **203.578.272.000.000.000 operaciones**.
Buscamos el mayor entero que cumpla:

$$n^3 \leq 203\,578\,272\,000\,000\,000$$

Matemáticamente:

$$n_{\max}=\left\lfloor\sqrt[3]{203\,578\,272\,000\,000\,000}\right\rfloor$$

La función siguiente obtiene la raíz cúbica entera mediante búsqueda binaria.
Así se evita decidir la frontera a partir de una potencia de coma flotante redondeada.
En cada paso se conserva un valor admisible por debajo y otro que supera el límite.

In [6]:
def raiz_cubica_entera(presupuesto):
    """Devuelve el mayor entero n tal que n**3 no supera el presupuesto.

    Se aplica búsqueda binaria. El límite inferior es un valor admisible
    y el superior uno que ya no lo es. En cada paso se descarta la mitad
    del intervalo. Al quedar dos enteros consecutivos, se devuelve el inferior.
    """
    if not isinstance(presupuesto, int) or presupuesto < 0:
        raise ValueError("El presupuesto debe ser un entero no negativo.")

    inferior = 0
    superior = presupuesto + 1

    while inferior + 1 < superior:
        medio = (inferior + superior) // 2
        # // es división entera: el candidato también debe ser entero.
        if medio**3 <= presupuesto:
            inferior = medio
        else:
            superior = medio

    return inferior

## 3. Calcular y verificar el máximo

Comprobar que un tamaño cabe no basta para demostrar que es el mayor.
También verificamos que el entero siguiente ya supera el presupuesto.

In [7]:
n_maximo = raiz_cubica_entera(presupuesto)
coste_n = n_maximo**3
coste_siguiente = (n_maximo + 1)**3

# Las dos desigualdades comprueban la frontera de la solución.
assert coste_n <= presupuesto
assert presupuesto < coste_siguiente

print("Mayor tamaño resoluble:", n_maximo)
print("Operaciones necesarias para ese tamaño:", coste_n)
print("Operaciones necesarias para el siguiente:", coste_siguiente)
print("¿Cabe el tamaño calculado?", coste_n <= presupuesto)
print("¿Supera el límite el siguiente?", coste_siguiente > presupuesto)

Mayor tamaño resoluble: 588270
Operaciones necesarias para ese tamaño: 203577653255283000
Operaciones necesarias para el siguiente: 203578691441826511
¿Cabe el tamaño calculado? True
¿Supera el límite el siguiente? True


## 4. Interpretar el resultado

**El tamaño máximo es $n=588\,270$.**

- $588\,270^3=203\,577\,653\,255\,283\,000$: cabe en el día disponible.
- $588\,271^3=203\,578\,691\,441\,826\,511$: lo supera.

El resultado supone una tasa constante y el coste exacto $n^3$.
Saber únicamente que un algoritmo es $O(n^3)$ no permite obtener este máximo exacto:
faltan las constantes, otras operaciones y las condiciones reales de ejecución.
El cuaderno resuelve el modelo matemático del ejercicio; no realiza una prueba de rendimiento del equipo.